# 01 · Machine learning: dataset → VIF → 7 models → leave-one-event-out

Input: `data/flood_dataset_all_events.csv` (download from Google Drive → `flood_project`).
All tables and figures are saved to `outputs/`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.config import DATASET_CSV, OUT, CAT, NUM, TARGET, SEED
from src.ml_utils import (add_features, usable_numeric, get_models, fit_model,
                          evaluate, leave_one_event_out)

df = add_features(pd.read_csv(DATASET_CSV))
print('Shape:', df.shape)
df.head()

## 1 · Data checks

In [ ]:
print('Missing values:', int(df.isna().sum().sum()))
print('\nSamples per event and class:')
display(pd.crosstab(df['event'], df[TARGET], margins=True))
df.describe().T.round(2)

In [ ]:
# Numeric factors that actually vary (drops constant columns such as a broken drain_dens)
NUM_USED = usable_numeric(df, NUM)
FEATURES = NUM_USED + CAT
print('Using', len(FEATURES), 'features:', FEATURES)

## 2 · Pearson correlation

In [ ]:
corr = df[NUM_USED].corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, square=True)
plt.title('Pearson correlation of conditioning factors')
plt.tight_layout(); plt.savefig(OUT / 'pearson_correlation.png', dpi=200); plt.show()

high = [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(NUM_USED)
        for b in NUM_USED[i + 1:] if abs(corr.loc[a, b]) >= 0.7]
print('Pairs with |r| >= 0.7:', high or 'none')

## 3 · VIF (multicollinearity)

In [ ]:
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

Xv = add_constant(pd.DataFrame(StandardScaler().fit_transform(df[NUM_USED]), columns=NUM_USED))
vif = pd.DataFrame({'factor': NUM_USED,
                    'VIF': [variance_inflation_factor(Xv.values, i + 1) for i in range(len(NUM_USED))]})
vif['TOL'] = 1 / vif['VIF']
vif['problem (VIF>10)'] = vif['VIF'] > 10
vif = vif.sort_values('VIF', ascending=False).round(3)
vif.to_csv(OUT / 'vif_table.csv', index=False)
vif

## 4 · Experiment 1 — random stratified 80/20 split (same as the base paper)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

MODELS = get_models()
train, test = train_test_split(df, test_size=0.2, stratify=df[TARGET], random_state=SEED)
print('Train:', len(train), ' Test:', len(test))

results, probs = [], {}
for name, model in MODELS.items():
    pipe = fit_model(model, train, train[TARGET], NUM_USED)
    probs[name] = pipe.predict_proba(test[FEATURES])[:, 1]
    results.append({'Model': name, **evaluate(test[TARGET], probs[name])})
    print(f'{name:9s} AUC = {results[-1]["AUC"]:.3f}')

res_random = pd.DataFrame(results).set_index('Model').round(3)
res_random.to_csv(OUT / 'results_random_split.csv')
res_random.sort_values('AUC', ascending=False)

In [ ]:
plt.figure(figsize=(7, 6))
for name, p in probs.items():
    fpr, tpr, _ = roc_curve(test[TARGET], p)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(test[TARGET], p):.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.title('ROC curves — random 80/20 split'); plt.legend(loc='lower right')
plt.tight_layout(); plt.savefig(OUT / 'roc_random_split.png', dpi=200); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (name, p) in zip(axes.ravel(), probs.items()):
    cm = confusion_matrix(test[TARGET], (p >= 0.5).astype(int))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Non-flood', 'Flood'], yticklabels=['Non-flood', 'Flood'])
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
axes.ravel()[-1].axis('off')
plt.tight_layout(); plt.savefig(OUT / 'confusion_matrices_random.png', dpi=200); plt.show()

## 5 · Experiment 2 — leave-one-event-out (train on 2 floods, test on the unseen 3rd)

In [ ]:
res_loeo = leave_one_event_out(df, MODELS, NUM_USED).round(3)
res_loeo.to_csv(OUT / 'results_leave_one_event_out.csv', index=False)

auc_table = res_loeo.pivot(index='Model', columns='Test event', values='AUC')
auc_table['Mean AUC'] = auc_table.mean(axis=1)
auc_table['Random split AUC'] = res_random['AUC']
auc_table = auc_table.sort_values('Mean AUC', ascending=False).round(3)
auc_table.to_csv(OUT / 'auc_summary.csv')
auc_table

## 6 · Experiment 3 — without absolute elevation
HAND already describes "height above the nearest stream". If performance holds without elevation,
the model can be applied to higher regions (Hyderabad ≈ 500 m) without being misled.

In [ ]:
NUM_NO_ELEV = [c for c in NUM_USED if c != 'elevation']
subset = {k: MODELS[k] for k in ['RF', 'CatBoost', 'XGBoost']}
res_ne = leave_one_event_out(df, subset, NUM_NO_ELEV)

no_elev = res_ne.pivot(index='Model', columns='Test event', values='AUC')
no_elev['Mean AUC (no elevation)'] = no_elev.mean(axis=1)
no_elev['Mean AUC (with elevation)'] = auc_table.loc[no_elev.index, 'Mean AUC']
no_elev = no_elev.round(3)
no_elev.to_csv(OUT / 'experiment3_no_elevation.csv')
no_elev

## 7 · Save the final models (trained on ALL data)
- `rf_all_features.pkl` — Random Forest with every factor
- `rf_no_elevation.pkl` — **final model** used for maps of new areas

In [ ]:
import json
rf_all = fit_model(MODELS['RF'], df, df[TARGET], NUM_USED)
rf_final = fit_model(MODELS['RF'], df, df[TARGET], NUM_NO_ELEV)

joblib.dump(rf_all, OUT / 'rf_all_features.pkl')
joblib.dump(rf_final, OUT / 'rf_no_elevation.pkl')
(OUT / 'model_features.json').write_text(json.dumps(
    {'rf_all_features': NUM_USED + CAT, 'rf_no_elevation': NUM_NO_ELEV + CAT}, indent=2))
print('Saved models to', OUT)